# 10 Working with Geostationary Sensor Datasets

# 10.1 Advanced Baseline Imager (ABI)

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
# import matplotlib.patches as patches
from cartopy import crs as ccrs
import cartopy.feature as cfeature
from pathlib import Path
# import datetime
import pandas as pd
# import scipy
import glob
from scipy.ndimage import distance_transform_edt
from satpy import Scene, DataQuery

In [ ]:
notebook_dir = Path.cwd()
fpath = notebook_dir / "data"

In [ ]:
# Optional: Uniformly make font sizes larger and set default figure size
plt.rcParams.update({"font.size": 16, 
    "figure.figsize": [12, 12],
    "font.sans-serif": ["Segoe UI", "DejaVu Sans", "sans-serif"]
    })

In [ ]:
filename1 = fpath / "abi" / "OR_ABI-L1b-RadF-M6C13_G19_s20251801520210_e20251801529530_c20251801529589.nc"
f1 = xr.open_dataset(filename1, engine="h5netcdf", mask_and_scale=True)
f1

In [ ]:
f1["Rad"].attrs

In [ ]:
def create_subsample(var, step=10):
  return np.array(var)[::step, ::step]

In [ ]:
skip = 20
rad = np.array(f1.Rad)
rad_sub = create_subsample(rad, skip)

In [ ]:
plt.figure()
plt.imshow(rad_sub, cmap="Greys")
plt.gca().set(frame_on=False, xticks=[], yticks=[])
plt.show()

In [ ]:
def get_sat_params(fid):
  geom = fid.goes_imager_projection

  # In meters and degrees
  satellite_params = {
  "longitude": geom.longitude_of_projection_origin,
  "latitude": 0.0,
  "height": geom.perspective_point_height,
  "req" : geom.semi_major_axis,
  "rpol" : geom.semi_minor_axis,
  "R_earth" : 6371000
  }

  return satellite_params

In [ ]:
satellite_params = get_sat_params(f1)

In [ ]:
print(satellite_params)

## 10.1.1 Unit Conversions

In [ ]:
# Simpler method of converting from radiance to reflectance for bands 1-6
# reflectance ρ𝑓v = kappa factor κ * radiance Lv
# kappa factor = ((πꞏd2)/Esun) represents the incident Lambertianequivalent radiance, where d is the instantaneous Earth-Sun distance (in Astronomical Units) and Esun is the
# solar irradiance in the respective bandpass (in W/(m2 µm)).
f1["kappa0"]

In [ ]:
# Kappa value instead of conversion code
reflectance_abi = f1["kappa0"] * f1["Rad"]

In [ ]:
# Converting radiance to brightness temperature (T)
# T = [ fk2 / (log((fk1 / L) + 1)) - bc1 ] / bc2
# L - radiance
# fk1 and fk2 are coefficients of the Planck function derived from physical constants (i.e., the speed of light, the Boltzmann constant, and the Planck constant) and the bandpass central wavenumber
# bc1 and bc2 are the spectral response function offset and scale correction terms
fk1 = f1["planck_fk1"]
fk2 = f1["planck_fk1"]
bc1 = f1["planck_bc1"]
bc2 = f1["planck_bc2"]

bt_abi = fk2 / np.log((fk1 / f1["Rad"]) + 1) - bc1 / bc2

## 10.1.1 Fixed Grid

In [ ]:
def get_lat_lon(fid, satellite_params):
  # Get x, y, and projection variables from the file
  x = fid["x"]
  y = fid["y"]

  # Extract geometry parameters
  req = satellite_params["req"]
  rpol = satellite_params["rpol"]
  H = satellite_params["height"] + satellite_params["req"]
  lamda0 = satellite_params["longitude"]*np.pi/180.0 # convert to radians

  # Eq. 10.7-10.9
  a = np.sin(x)**2 + np.cos(x)**2 * \
    (np.cos(y)**2 + (req**2/rpol**2)*np.sin(y)**2)
  b = -2 * H * np.cos(x) * np.cos(y)
  c = H**2 - req**2

  # Distance from satellite to pixel
  # Eq. 10.6
  rs = (-b - np.sqrt(b**2 - 4*a*c) ) / (2*a)

  # Eq. 10.3-10.5
  sx = rs*np.cos(x)*np.cos(y)
  sy = -rs*np.sin(x)
  sz = rs*np.cos(x)*np.sin(y)

  # Eq. 10.1-10.2
  # Convert into latitude and longitude (radians)
  latitude = np.arctan(req**2/rpol**2 * \
    sz / np.sqrt((H-sx)**2 + sy**2))
  longitude = lamda0 - np.arctan(sy/(H-sx))

  return np.array(latitude.T*180.0/np.pi), np.array(longitude.T*180.0/np.pi)

In [ ]:
lats, lons = get_lat_lon(f1, satellite_params)

In [ ]:
def fill_nan_nearest_fast(data, max_distance=None):

    filled_data = data.copy()
    nan_mask = np.isnan(data)

    if not np.any(nan_mask):
        return filled_data

    # Create distance map from valid data points
    valid_mask = ~nan_mask
    distances, indices = distance_transform_edt(nan_mask, return_indices=True)

    # Apply distance constraint if specified
    if max_distance is not None:
        fill_mask = nan_mask & (distances <= max_distance)
    else:
        fill_mask = nan_mask

    # Fill NaN values with nearest valid values
    filled_data[fill_mask] = data[indices[0][fill_mask], indices[1][fill_mask]]

    return filled_data

In [ ]:
# Subsample to display faster
lats_sub = create_subsample(lats, skip)
lons_sub = create_subsample(lons, skip)

# Remove nans to use pcolormesh
lats_filled = fill_nan_nearest_fast(lats_sub)
lons_filled = fill_nan_nearest_fast(lons_sub)
# rad_sub_masked = np.ma.masked_where(np.isnan(rad_sub), rad_sub)

In [ ]:
proj=ccrs.Orthographic(central_longitude=satellite_params["longitude"])
# proj=ccrs.PlateCarree()
proj_from = ccrs.PlateCarree()

In [ ]:
plt.figure(figsize=[8,8])
ax = plt.subplot(projection=proj)
ax.coastlines(color="yellow")
ax.pcolormesh(lons_filled, lats_filled, rad_sub, cmap="Greys", transform=proj_from)
plt.show()

##10.1.2 Viewing Geometry

In [ ]:
def calculate_goes_viewing_angles(lons, lats, satellite_params):
  # Satellite parameters
  sat_lon = satellite_params["longitude"]
  sat_lat = satellite_params["latitude"]
  sat_height = satellite_params["height"]/1000
  r_earth = satellite_params["R_earth"]/1000
  H = (r_earth + sat_height)

  # Convert to radians
  lon_rad = np.radians(lons)
  lat_rad = np.radians(lats)
  sat_lon_rad = np.radians(sat_lon)
  sat_lat_rad = np.radians(sat_lat)

  # Calculate zenith angle
  delta_lon = lon_rad - sat_lon_rad

  # Eq. 10.11
  central_angle = np.arccos(
      np.sin(lat_rad) * np.sin(sat_lat_rad) +
      np.cos(lat_rad) * np.cos(sat_lat_rad) * np.cos(delta_lon)
  )

  # 10.12
  sat_to_ground_dist = np.sqrt(r_earth**2 + H**2 - 2*r_earth*H*np.cos(central_angle))

  # Eq. 10.9
  sin_zenith = (H * np.sin(central_angle)) / sat_to_ground_dist
  sin_zenith = np.clip(sin_zenith, 0, 1)

  # Eq. 10.10
  view_zenith = np.degrees(np.arcsin(sin_zenith))

  # Calculate azimuth angle
  delta_lon_deg = np.degrees(delta_lon)
  delta_lat_deg = lats - sat_lat

  # Calculate azimuth from ground point toward satellite sub-point
  y = -np.sin(delta_lon) * np.cos(lat_rad)
  x = (np.cos(lat_rad) * np.sin(sat_lat_rad) -
        np.sin(lat_rad) * np.cos(sat_lat_rad) * np.cos(delta_lon))

  azimuth_to_sat = np.degrees(np.arctan2(y, x))

  # Eq. 10.13
  view_azimuth = np.degrees(np.arctan2(y, x))

  # Convert to -180° to +180°,
  view_azimuth = (view_azimuth + 180) % 360

  # Convert to North = 0, East = 90, South = 180, West = -90
  view_azimuth = np.where(view_azimuth > 180, view_azimuth - 360, view_azimuth)

  # Set azimuth to 0 directly below satellite
  at_subsat_point = (central_angle < 1e-6)
  view_azimuth = np.where(at_subsat_point, 0.0, view_azimuth)

  return view_zenith, view_azimuth

In [ ]:
vza, vaa = calculate_goes_viewing_angles(lons_sub, lats_sub, satellite_params)

In [ ]:
np.nanmax(vza)

In [ ]:
fig, ax = plt.subplots(2,2, figsize=[8,8])

img = ax[0,0].imshow(vza, vmin=0, vmax=90)
fig.colorbar(img)
ax[0,0].set_title("Viewing Zenith")

img = ax[0,1].imshow(vaa, vmin=-180, vmax=180, cmap="RdYlBu_r")
fig.colorbar(img)
ax[0,1].set_title("Viewing Azimuth")

img = ax[1,0].imshow(lons_sub, vmin=-180, vmax=0,cmap="RdYlBu_r")
fig.colorbar(img)
ax[1,0].set_title("Longitude")

img = ax[1,1].imshow(lats_sub, vmin=-90, vmax=90, cmap="RdYlBu_r")
fig.colorbar(img)
ax[1,1].set_title("Latitude")

for ax0 in ax.ravel():
  ax0.set(frame_on=True, xticks=[], yticks=[])

plt.show()

## 10.1.3 Parallax Correction

In [ ]:
def get_parallax_lat_lons(lats, cloud_heights, surface_heights, vza, vaa):
  vaa_rad = np.deg2rad(vaa)
  vza_rad = np.deg2rad(vza)

  dpm_lat = 8.9932e-6
  dpm_lon = dpm_lat/np.cos(np.deg2rad(lats))

  delta_height = cloud_heights - surface_heights

  delta_lon = delta_height*np.tan(vza_rad)*np.sin(vaa_rad)*dpm_lon
  delta_lat = delta_height*np.tan(vza_rad)*np.cos(vaa_rad)*dpm_lat

  return delta_lon, delta_lat

In [ ]:
cloud_ht = 15240
sea_level = 0

In [ ]:
parallax_lon, parallax_lat = get_parallax_lat_lons(lats_sub, cloud_ht, sea_level, vza, vaa)

In [ ]:
lons_vector = create_subsample(lons_sub, 10)
lats_vector = create_subsample(lats_sub, 10)
parallax_lon_vector = create_subsample(parallax_lon, 10)
parallax_lat_vector = create_subsample(parallax_lat, 10)

In [ ]:
def get_extent(lons, lats):
  lon_min, lon_max = np.nanmin(lons), np.nanmax(lons)
  lat_min, lat_max = np.nanmin(lats), np.nanmax(lats)

  return [lon_min, lon_max, lat_min, lat_max]

In [ ]:
fig = plt.figure(figsize=(12, 12))

ax1 = fig.add_subplot(1, 1, 1, projection=proj)
ax1.set_global()
ax1.add_feature(cfeature.COASTLINE, linewidth=0.8)

ax1.pcolormesh(lons_filled, lats_filled, rad_sub, cmap="Greys_r",
  transform=proj_from, alpha=0.75)

magnitude = np.sqrt(parallax_lon_vector**2 + parallax_lat_vector**2)
magnitude = np.clip(magnitude, 0, 0.5)

quiver = ax1.quiver(lons_vector, lats_vector,
  parallax_lon_vector, parallax_lat_vector,
  magnitude, scale=1 / 0.2, width=0.005,
  transform=proj_from)

plt.colorbar(quiver, ax=ax1, shrink=0.7, pad=0.02,
  orientation="horizontal", label="Parallax Correction (°)")

plt.show()

# 10.2 Spinning Enhanced Visible and Infrared Imager (SEVIRI)


In [ ]:
# SEVIRI
filename_sev = [fpath / "seviri" / "MSG3-SEVI-MSG15-0100-NA-20240920121242.473000000Z-NA.nat"]
scn_sev = Scene(reader="seviri_l1b_native", filenames=filename_sev)

# Scan the file contents
print(scn_sev.available_dataset_names())

In [ ]:
scn_sev.load(["VIS006"], upper_right_corner="NE")
print(scn_sev["VIS006"])

In [ ]:
rad_sev_vis006 = np.array(scn_sev["VIS006"])

In [ ]:
# Make readable labels
time_str = scn_sev["VIS006"].time_parameters["nominal_start_time"].strftime("%m-%d-%Y %H:%M UTC")

title_sev= f"{scn_sev["VIS006"].platform_name} {time_str}"
label_sev = f"{str(scn_sev["VIS006"].wavelength.central)+scn_sev["VIS006"].wavelength.unit} {scn_sev["VIS006"].standard_name.replace("_", " ")} ({scn_sev["VIS006"].units})"

title_sev = title_sev.encode("ascii", "ignore").decode("ascii")
label_sev = label_sev.encode("ascii", "ignore").decode("ascii")

title_sev, label_sev

In [ ]:
ax = plt.subplot()

img = plt.imshow(rad_sev_vis006, cmap="Greys_r")
ax.set(frame_on=True, xticks=[], yticks=[])

plt.colorbar(img, label=label_sev)
plt.title(title_sev)
plt.show()

In [ ]:
lon_sev, lat_sev = scn_sev["VIS006"].area.get_lonlats()

In [ ]:
scn_sev["VIS006"].attrs["orbital_parameters"]

In [ ]:
# Defien geo coord. ref. system
sat_height = scn_sev["VIS006"].attrs["orbital_parameters"]["satellite_actual_altitude"]
img_proj = ccrs.Geostationary(satellite_height=sat_height)

# Define extent
sev_x = np.array(scn_sev["VIS006"].x)
sev_y = np.array(scn_sev["VIS006"].y)
img_extent = (sev_x.min(), sev_x.max(), sev_y.min(), sev_y.max())

In [ ]:
plt.figure()
ax = plt.subplot(projection=img_proj)
ax.coastlines(color="yellow", alpha=0.5, linewidth=2.0)
ax.set_global()

img = plt.imshow(rad_sev_vis006, cmap="Greys_r", extent=img_extent, transform=img_proj, origin="upper")
plt.colorbar(img, label=label_sev)
plt.title(title_sev)

# ax.set(frame_on=True, xticks=[], yticks=[])

plt.show()

# 10.3 Advanced Himawari Imager (AHI)

In [ ]:
# AHI
ahi_dir = fpath / "ahi"
filenames2 = list(ahi_dir.glob("HS_*"))
scn_ahi = Scene(filenames=filenames2, reader="ahi_hsd")
scn_ahi.available_dataset_names()

In [ ]:
fillval = -9999.9

In [ ]:
scn_ahi.available_dataset_names()

In [ ]:
# Load the data in radiance units
band_ahi = scn_ahi.available_dataset_ids()[1]["name"]
band_ahi

In [ ]:
channel_id_ahi = DataQuery(name = "B01")
channel_id_ahi

In [ ]:
scn_ahi.load(["B01"])

In [ ]:
band_ahi = scn_ahi.available_dataset_ids()[0]["name"]
channel_id_ahi = DataQuery(name="B01", calibration="radiance")

In [ ]:
scn_ahi.load([channel_id_ahi])
scan_band_ahi = scn_ahi[channel_id_ahi]

In [ ]:
rad_ahi = np.array(getattr(scan_band_ahi,"data"))

In [ ]:
start_time_ahi = getattr(scan_band_ahi,"start_time")
units_ahi = getattr(scan_band_ahi,"units")
wavelength_ahi = getattr(scan_band_ahi,"wavelength")

In [ ]:
time_str_ahi = start_time_ahi.strftime("%m-%d-%Y %H:%M UTC")

title_ahi = f"{scan_band_ahi.platform_name} {scan_band_ahi.sensor.upper()} {time_str_ahi}"
label_ahi = f"{str(scan_band_ahi.wavelength.central)+scan_band_ahi.wavelength.unit} {scan_band_ahi.standard_name.replace("_", " ")} ({scan_band_ahi.units})"
title_ahi, label_ahi

In [ ]:
# Get geostarionary coord. ref. system
sat_height = scan_band_ahi.attrs["orbital_parameters"]["satellite_actual_altitude"]
projection_longitude = scan_band_ahi .attrs["orbital_parameters"]["projection_longitude"]

img_proj = ccrs.Geostationary(satellite_height=sat_height, central_longitude=projection_longitude)
img_extent = (-5570248.4773, 5567248.0742, -5567248.0742, 5570248.4773)

In [ ]:
# Get extent
ahi_x = np.array(scan_band_ahi.x)
ahi_y = np.array(scan_band_ahi.y)
img_extent = (ahi_x.min(), ahi_x.max(), ahi_y.min(), ahi_y.max())

In [ ]:
plt.figure(figsize=[10,10])
ax = plt.subplot(projection=img_proj)
ax.coastlines(color="yellow", alpha=0.5, linewidth=1)
ax.set_global()

# If you don"t subset AHI, ram is exceeded in Colab
img = plt.imshow(rad_ahi[::10,::10], cmap="Greys_r", extent=img_extent, transform=img_proj, origin="upper", vmin=0, vmax=300)
plt.colorbar(img, label=label_ahi)
plt.title(title_ahi)

ax.set(frame_on=True, xticks=[], yticks=[])

plt.show()